# HW 7 — Chapter 8: Data Modification
**Group:** EOS_grp_2  
**Database:** Northwinds2024Student  
**Author:** Prabhjot Kaur 
**Date:** April 13, 2026  

---

## Overview
This notebook contains 10 original propositions and SQL queries covering INSERT, UPDATE, DELETE, and Transactions from Chapter 8 — Data Modification.  


---
## Proposition 1 — INSERT a new customer into Sales.Customer
**Problem:** The sales team has signed a new corporate client based in New York. A record must be added to the Sales.Customer table so the company can begin accepting and tracking their orders.

**Why special:** This is the foundational INSERT operation a new customer must exist in Sales.Customer before any orders can be placed. 

In [ ]:
INSERT INTO Sales.Customer (
    CustomerCompanyName,
    CustomerContactName,
    CustomerContactTitle,
    CustomerAddress,
    CustomerCity,
    CustomerPostalCode,
    CustomerCountry,
    CustomerPhoneNumber
)
VALUES (
    'Metro Supplies Inc.',
    'Johny Doe',
    'Purchasing Manager',
    '45 Broadway Ave',
    'New York',
    '10006',
    'USA',
    '212-555-0187'
);
SELECT CustomerCompanyName, CustomerContactName, CustomerCity, CustomerCountry
FROM Sales.Customer
WHERE CustomerCompanyName = 'Metro Supply Inc.';

---
## Proposition 2 — INSERT a new employee then immediately assign them an order (Transaction)
**Problem:** A new sales representative has been hired. Their record must be inserted into HumanResources.Employee and an existing unassigned order must be immediately reassigned to them
**Why special:** Uses SCOPE_IDENTITY() to capture the auto-generated EmployeeId and immediately use it in an UPDATE on Sales.Order — demonstrating INSERT + UPDATE 

In [ ]:
BEGIN TRANSACTION;

INSERT INTO HumanResources.Employee (
    EmployeeLastName,
    EmployeeFirstName,
    EmployeeTitle,
    EmployeeTitleOfCourtesy,
    BirthDate,
    HireDate,
    EmployeeAddress,
    EmployeeCity,
    EmployeeRegion,
    EmployeePostalCode,
    EmployeeCountry,
    EmployeePhoneNumber
)
VALUES (
    'Rivera',
    'Sofia',
    'Sales Representative',
    'Ms.',
    '1990-06-15',
    GETDATE(),
    '123 Main Street',
    'Seattle',
    'WA',
    '98101',
    'USA',
    '206-555-0198'
);

DECLARE @NewEmpId INT = SCOPE_IDENTITY();

UPDATE Sales.[Order]
SET EmployeeId = @NewEmpId
WHERE OrderId = (
    SELECT TOP 1 OrderId
    FROM Sales.[Order]
    ORDER BY OrderDate DESC
);

SELECT o.OrderId, o.EmployeeId, e.EmployeeFirstName, e.EmployeeLastName
FROM Sales.[Order] o
JOIN HumanResources.Employee e ON o.EmployeeId = e.EmployeeId
WHERE o.EmployeeId = @NewEmpId;

ROLLBACK;
PRINT 'Proposition 2 complete. No permanent changes made.';


---
## Proposition 3 — UPDATE employee title based on order volume performance
**Problem:** Management wants to promote employees who have handled more than 50 orders to 'Senior Sales Representative'. The HumanResources.Employee table must be updated to reflect this title change.

**Why special:** Uses a subquery against Sales.Order grouped by EmployeeId with HAVING COUNT > 50 inside an UPDATE — a performance-driven, data-backed title promotion with no hardcoded employee IDs.

In [ ]:
-- Preview employees who qualify
SELECT e.EmployeeId,
       e.EmployeeFirstName,
       e.EmployeeLastName,
       e.EmployeeTitle,
       COUNT(o.OrderId) AS TotalOrders
FROM HumanResources.Employee e
JOIN Sales.[Order] o ON e.EmployeeId = o.EmployeeId
GROUP BY e.EmployeeId, e.EmployeeFirstName, e.EmployeeLastName, e.EmployeeTitle
HAVING COUNT(o.OrderId) > 50;

UPDATE HumanResources.Employee
SET EmployeeTitle = 'Senior Sales Representative'
WHERE EmployeeId IN (
    SELECT EmployeeId
    FROM Sales.[Order]
    GROUP BY EmployeeId
    HAVING COUNT(OrderId) > 50
);

SELECT EmployeeId, EmployeeFirstName, EmployeeLastName, EmployeeTitle
FROM HumanResources.Employee
WHERE EmployeeTitle = 'Senior Sales Representative';


---
## Proposition 4 — UPDATE Sales.Order freight cost for international shipments
**Problem:** Due to rising shipping costs, all orders being shipped to countries outside the USA must have their Freight amount increased by 20% to offset carrier surcharges.

**Why special:** Directly updates the Freight (Currency/money) column in Sales.Order using a WHERE filter on ShipToCountry — a realistic pricing adjustment that targets a specific subset of orders without touching domestic shipments.

In [ ]:
BEGIN TRANSACTION;

-- Preview before update
SELECT ShipToCountry,
       COUNT(OrderId) AS TotalOrders,
       AVG(Freight) AS AvgFreightBefore
FROM Sales.[Order]
WHERE ShipToCountry <> 'USA'
GROUP BY ShipToCountry
ORDER BY AvgFreightBefore DESC;

UPDATE Sales.[Order]
SET Freight = Freight * 1.20
WHERE ShipToCountry <> 'USA';

-- Preview after update
SELECT ShipToCountry,
       COUNT(OrderId) AS TotalOrders,
       AVG(Freight) AS AvgFreightAfter
FROM Sales.[Order]
WHERE ShipToCountry <> 'USA'
GROUP BY ShipToCountry
ORDER BY AvgFreightAfter DESC;

---
## Proposition 5 — UPDATE customer city to match their most recent ShipToCity
**Problem:** Some records in Sales.Customer have outdated city information. The CustomerCity field should be synced to the ShipToCity from each customer's most recently placed order to reflect their current location.

**Why special:** Uses a correlated subquery with TOP 1 ... ORDER BY OrderDate DESC inside an UPDATE statement — a data quality fix that pulls from transactional history to correct master records.

In [ ]:
BEGIN TRANSACTION;

SELECT c.CustomerId,
       c.CustomerCompanyName,
       c.CustomerCity AS CurrentCity,
       (
           SELECT TOP 1 o.ShipToCity
           FROM Sales.[Order] o
           WHERE o.CustomerId = c.CustomerId
           ORDER BY o.OrderDate DESC
       ) AS LatestShipToCity
FROM Sales.Customer c
WHERE c.CustomerCity <> (
    SELECT TOP 1 o.ShipToCity
    FROM Sales.[Order] o
    WHERE o.CustomerId = c.CustomerId
    ORDER BY o.OrderDate DESC
);

UPDATE Sales.Customer
SET CustomerCity = (
    SELECT TOP 1 o.ShipToCity
    FROM Sales.[Order] o
    WHERE o.CustomerId = Sales.Customer.CustomerId
    ORDER BY o.OrderDate DESC
)
WHERE CustomerId IN (
    SELECT DISTINCT CustomerId FROM Sales.[Order]
);

SELECT TOP 5 CustomerId, CustomerCompanyName, CustomerCity
FROM Sales.Customer
ORDER BY CustomerId;



---
## Proposition 6 — UPDATE Sales.OrderDetail discount for high-volume order lines
**Problem:** As a bulk purchase incentive, management wants to apply a 10% discount to all Sales.OrderDetail lines where the Quantity ordered exceeds 40 units and no discount currently exists.

**Why special:** Targets the Discount column in Sales.OrderDetail with a compound WHERE condition — only affects qualifying high-quantity lines with zero discount, avoiding double-discounting existing promotional lines.

In [ ]:

BEGIN TRANSACTION;

-- Preview qualifying lines
SELECT OrderId, ProductId, Quantity, DiscountPercentage
FROM Sales.OrderDetail
WHERE Quantity > 40
AND DiscountPercentage = 0;

UPDATE Sales.OrderDetail
SET DiscountPercentage = 0.10
WHERE Quantity > 40
AND DiscountPercentage = 0;

-- Verify
SELECT OrderId, ProductId, Quantity, DiscountPercentage
FROM Sales.OrderDetail
WHERE Quantity > 40
AND DiscountPercentage = 0.10;


---
## Proposition 7 — DELETE Sales.OrderDetail lines for discontinued products on old orders
**Problem:** Before archiving historical data, all Sales.OrderDetail records linked to discontinued products on orders placed before 1997 should be removed to reduce data clutter and improve archive quality.

**Why special:** Multi-table DELETE using JOIN across Sales.OrderDetail, Production.Product, and Sales.Order — simultaneously filters by product discontinuation status and order date. A real-world pre-archive cleanup pattern.

In [ ]:
BEGIN TRANSACTION;

-- Preview rows to be deleted
SELECT od.OrderId, p.ProductName, o.OrderDate
FROM Sales.OrderDetail od
JOIN Production.Product p ON od.ProductId = p.ProductId
JOIN Sales.[Order] o ON od.OrderId = o.OrderId
WHERE p.Discontinued = 1
AND o.OrderDate < '1997-01-01';

DELETE od
FROM Sales.OrderDetail od
JOIN Production.Product p ON od.ProductId = p.ProductId
JOIN Sales.[Order] o ON od.OrderId = o.OrderId
WHERE p.Discontinued = 1
AND o.OrderDate < '1997-01-01';

SELECT @@ROWCOUNT AS RowsDeleted;

---
## Proposition 8 — DELETE customers from Sales.Customer with no order history
**Problem:** The Sales.Customer table contains records for contacts who never placed a single order. These dormant records should be identified and removed to keep the customer database clean and accurate.

**Why special:** Uses NOT IN with a subquery against Sales.Order — safely identifies and removes zero-transaction customers with no risk of orphaning order data since these customers have none.

In [ ]:


BEGIN TRANSACTION;

-- Preview dormant customers
SELECT CustomerId, CustomerCompanyName, CustomerCity, CustomerCountry
FROM Sales.Customer
WHERE CustomerId NOT IN (
    SELECT DISTINCT CustomerId FROM Sales.[Order]
);

DELETE FROM Sales.Customer
WHERE CustomerId NOT IN (
    SELECT DISTINCT CustomerId FROM Sales.[Order]
);

SELECT @@ROWCOUNT AS RowsDeleted;


---
## Proposition 9 — Transaction: Reassign all orders then DELETE a departing employee
**Problem:** An employee is leaving the company. Before their record can be deleted from HumanResources.Employee, all their assigned orders in Sales.Order must be transferred to another employee — both steps wrapped in a transaction to prevent orphaned foreign key references.

**Why special:** Models a complete HR offboarding workflow in SQL — UPDATE then DELETE with a verification SELECT between steps. Demonstrates referential integrity awareness and real-world use of transactions.

In [ ]:

BEGIN TRANSACTION;

-- Step 1: Reassign all orders from employee 9 to employee 2
UPDATE Sales.[Order]
SET EmployeeId = 2
WHERE EmployeeId = 9;

-- Step 2: Verify no orders remain under employee 9
SELECT COUNT(*) AS RemainingOrders
FROM Sales.[Order]
WHERE EmployeeId = 9;

-- Step 3: Delete the employee record
DELETE FROM HumanResources.Employee
WHERE EmployeeId = 9;

-- Step 4: Confirm employee is gone
SELECT COUNT(*) AS EmployeeStillExists
FROM HumanResources.Employee
WHERE EmployeeId = 9;




---
## Proposition 10 — Transaction: Preview revenue impact of VIP discount then ROLLBACK
**Problem:** Management wants to evaluate the financial impact of applying a 25% loyalty discount to all Sales.OrderDetail lines belonging to the highest-spending customer — without permanently committing any changes to the database.

**Why special:** Uses a nested subquery to dynamically identify the top customer by lifetime revenue, applies the discount, previews the net revenue result across the top 5 customers, then rolls back entirely. A sophisticated what-if financial simulation using transactions.

In [ ]:

BEGIN TRANSACTION;

-- Step 1: Apply 25% discount to top customer's order lines
UPDATE od
SET od.DiscountPercentage = 0.25
FROM Sales.OrderDetail od
JOIN Sales.[Order] o ON od.OrderId = o.OrderId
WHERE o.CustomerId = (
    SELECT TOP 1 o2.CustomerId
    FROM Sales.[Order] o2
    JOIN Sales.OrderDetail od2 ON o2.OrderId = od2.OrderId
    GROUP BY o2.CustomerId
    ORDER BY SUM(od2.UnitPrice * od2.Quantity) DESC
);

-- Step 2: Preview net revenue for top 5 customers after discount
SELECT TOP 5
    c.CustomerCompanyName,
    c.CustomerCountry,
    SUM(od.UnitPrice * od.Quantity * (1 - od.DiscountPercentage)) AS PreviewNetRevenue
FROM Sales.Customer c
JOIN Sales.[Order] o ON c.CustomerId = o.CustomerId
JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
GROUP BY c.CustomerCompanyName, c.CustomerCountry
ORDER BY PreviewNetRevenue DESC;


---
# Excel Power Query Reports — Sales and Revenue Trends
The following 5 queries pull from the actual Northwinds2024Student schema for use in Excel Power Query reports.

## Excel Report 1 — Employee Revenue Contribution
**Question:** Which sales reps drive the most revenue and what is their average order value?

**Power Query steps:** Rename columns, format TotalRevenue and AvgOrderValue as Currency, add Revenue Rank index column, create bar chart by employee name.

In [ ]:
-- Excel Report 1: Employee Revenue Contribution
-- Database: Northwinds2024Student

SELECT
    e.EmployeeFirstName + ' ' + e.EmployeeLastName AS Employee,
    e.EmployeeTitle,
    COUNT(DISTINCT o.OrderId) AS TotalOrders,
    SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)) AS TotalRevenue,
    SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)) / COUNT(DISTINCT o.OrderId) AS AvgOrderValue
FROM HumanResources.Employee e
JOIN Sales.[Order] o ON e.EmployeeId = o.EmployeeId
JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
GROUP BY e.EmployeeFirstName, e.EmployeeLastName, e.EmployeeTitle
ORDER BY TotalRevenue DESC;

## Excel Report 2 — Monthly Revenue Trend by Year
**Question:** How did total revenue trend month over month across all years in the database?

**Power Query steps:** Merge Year and Month columns into a Date label like 1996-07, change type to text, sort ascending, create line chart to show revenue trend over time.

In [ ]:
-- Excel Report 2: Monthly Revenue Trend by Year
-- Database: Northwinds2024Student

SELECT
    YEAR(o.OrderDate) AS [Year],
    MONTH(o.OrderDate) AS [Month],
    COUNT(DISTINCT o.OrderId) AS TotalOrders,
    SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)) AS MonthlyRevenue
FROM Sales.[Order] o
JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
GROUP BY YEAR(o.OrderDate), MONTH(o.OrderDate)
ORDER BY [Year], [Month];

## Excel Report 3 — Top 10 Customers by Lifetime Revenue
**Question:** Who are the highest-spending customers and how much have they contributed in net revenue?

**Power Query steps:** Format LifetimeRevenue as currency, add Rank index column starting at 1, add conditional column for segment (High = over 10000, Mid = 5000-10000, Low = under 5000), apply color coding by segment.

In [ ]:
-- Excel Report 3: Top 10 Customers by Lifetime Revenue
-- Database: Northwinds2024Student

SELECT TOP 10
    c.CustomerCompanyName,
    c.CustomerCity,
    c.CustomerCountry,
    COUNT(DISTINCT o.OrderId) AS TotalOrders,
    SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)) AS LifetimeRevenue
FROM Sales.Customer c
JOIN Sales.[Order] o ON c.CustomerId = o.CustomerId
JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
GROUP BY c.CustomerCompanyName, c.CustomerCity, c.CustomerCountry
ORDER BY LifetimeRevenue DESC;

## Excel Report 4 — Revenue by Customer Country
**Question:** Which countries generate the most revenue and how many customers does each country have?

**Power Query steps:** Format TotalRevenue as currency, sort descending, create a filled map chart or horizontal bar chart by country.

In [ ]:
-- Excel Report 4: Revenue by Customer Country
-- Database: Northwinds2024Student

SELECT
    c.CustomerCountry,
    COUNT(DISTINCT c.CustomerId) AS TotalCustomers,
    COUNT(DISTINCT o.OrderId) AS TotalOrders,
    SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)) AS TotalRevenue,
    SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)) / COUNT(DISTINCT o.OrderId) AS AvgOrderValue
FROM Sales.Customer c
JOIN Sales.[Order] o ON c.CustomerId = o.CustomerId
JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
GROUP BY c.CustomerCountry
ORDER BY TotalRevenue DESC;

## Excel Report 5 — Customer Lifetime Value with Primary Employee Attribution
**Question:** For each customer, what is their total lifetime revenue and which employee handled the most of their orders?

**Power Query steps:** Format LifetimeRevenue as currency, add a conditional segment column (High / Mid / Low), apply color-coded conditional formatting rows by segment value.

In [ ]:
-- Excel Report 5: Customer Lifetime Value with Primary Employee Attribution
-- Database: Northwinds2024Student

SELECT
    c.CustomerCompanyName,
    c.CustomerCity,
    c.CustomerCountry,
    COUNT(DISTINCT o.OrderId) AS TotalOrders,
    SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)) AS LifetimeRevenue,
    (
        SELECT TOP 1
            e2.EmployeeFirstName + ' ' + e2.EmployeeLastName
        FROM Sales.[Order] o2
        JOIN HumanResources.Employee e2 ON o2.EmployeeId = e2.EmployeeId
        WHERE o2.CustomerId = c.CustomerId
        GROUP BY e2.EmployeeFirstName, e2.EmployeeLastName
        ORDER BY COUNT(*) DESC
    ) AS PrimaryEmployee
FROM Sales.Customer c
JOIN Sales.[Order] o ON c.CustomerId = o.CustomerId
JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
GROUP BY c.CustomerCompanyName, c.CustomerCity, c.CustomerCountry
ORDER BY LifetimeRevenue DESC;